In [ ]:
# Aim: Initialize the isolated Lasso experiment with the exact DSN data and locked validation protocol.

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# Reproducibility
RANDOM_STATE = 42

# Load the competition-ready datasets
train_final = pd.read_csv("../data/train_final.csv")
test_final = pd.read_csv("../data/test_final.csv")

# Target
target = "total_sales"

# Exact locked DSN predictor set
lasso_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

# Exact categorical / numerical split used in our previous Lasso experiment
lasso_categorical = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

lasso_numeric = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

# Build X/y
X = train_final[lasso_features].copy()
y = train_final[target].copy()
X_test = test_final[lasso_features].copy()

# Locked validation protocol
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Train shape :", X.shape)
print("Test shape  :", X_test.shape)
print("Target shape:", y.shape)
print("\nCategorical features:")
print(lasso_categorical)
print("\nNumerical features:")
print(lasso_numeric)

print("\nFold sizes:")
for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(
        f"F{fold}: "
        f"train={len(train_idx):,}, "
        f"val={len(val_idx):,}"
    )

In [ ]:
# Aim: Establish a leakage-safe one-hot linear baseline with standardized numerical features.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso

# OOF predictions
lasso_oof = np.full(len(X), np.nan)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Fold-specific preprocessing
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ])

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("categorical", categorical_pipeline, lasso_categorical),
        ("numeric", numeric_pipeline, lasso_numeric)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("lasso", Lasso(
            alpha=1.0,
            max_iter=10000,
            random_state=RANDOM_STATE
        ))
    ])

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    lasso_oof[val_idx] = val_pred

    fold_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    fold_results.append(fold_rmse)

    print(f"F{fold} RMSE: {fold_rmse:.6f}")

# Overall OOF RMSE
lasso_oof_rmse = np.sqrt(
    mean_squared_error(y, lasso_oof)
)

print("\nFold mean RMSE:", np.mean(fold_results))
print("OOF RMSE      :", lasso_oof_rmse)
print("OOF complete  :", np.isfinite(lasso_oof).all())

In [ ]:
# Aim: Construct a compact interaction-ready feature space while preserving the original validation protocol.

X_poly = X.copy()

# Derive a compact product-family representation from the product identifier.
X_poly["product_family"] = (
    X_poly["product_code"]
    .str.replace("PRD-", "", regex=False)
    .str[0]
)

poly_categorical = [
    "product_family",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

poly_numeric = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

print("Feature shape:", X_poly.shape)
print("\nCategorical cardinalities:")
print(X_poly[poly_categorical].nunique(dropna=False))

print("\nNumerical features:")
print(poly_numeric)

In [ ]:
# Aim: Expand the compact feature space with second-order interactions for nonlinear linear-model fitting.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures

poly_preprocessor = ColumnTransformer([
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ))
        ]),
        poly_categorical
    ),
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        poly_numeric
    )
])

X_poly_encoded = poly_preprocessor.fit_transform(X_poly)

print("Encoded shape:", X_poly_encoded.shape)
print("Encoded type :", type(X_poly_encoded).__name__)

poly_features = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_poly_expanded = poly_features.fit_transform(X_poly_encoded)

print("Expanded shape:", X_poly_expanded.shape)
print("Expanded type :", type(X_poly_expanded).__name__)

In [ ]:
# Aim: Quantify the relationship between product price and total sales across the observed price range.

price_analysis = (
    train_final
    .assign(
        price_bin=pd.qcut(
            train_final["product_price"],
            q=10,
            duplicates="drop"
        )
    )
    .groupby("price_bin", observed=True)
    .agg(
        observations=("total_sales", "size"),
        mean_price=("product_price", "mean"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        std_sales=("total_sales", "std"),
        mean_residual_from_global_mean=(
            "total_sales",
            lambda s: s.mean() - y.mean()
        )
    )
    .reset_index()
)

print(price_analysis.to_string(index=False))

print("\nCorrelations:")
print(
    "Pearson  :", 
    train_final["product_price"].corr(train_final["total_sales"], method="pearson")
)
print(
    "Spearman :", 
    train_final["product_price"].corr(train_final["total_sales"], method="spearman")
)

In [ ]:
# Aim: Examine whether the price-sales relationship changes systematically across product categories.

category_price_profile = (
    train_final
    .groupby("product_category_clean")
    .agg(
        observations=("total_sales", "size"),
        mean_price=("product_price", "mean"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        sales_std=("total_sales", "std"),
        price_sales_corr=(
            "product_price",
            lambda s: s.corr(
                train_final.loc[s.index, "total_sales"],
                method="pearson"
            )
        )
    )
    .sort_values("mean_sales", ascending=False)
    .reset_index()
)

print(category_price_profile.to_string(index=False))

In [ ]:
# Aim: Examine whether the price-sales relationship varies across store formats.

format_price_profile = (
    train_final
    .groupby("store_format")
    .agg(
        observations=("total_sales", "size"),
        mean_price=("product_price", "mean"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        sales_std=("total_sales", "std"),
        price_sales_corr=(
            "product_price",
            lambda s: s.corr(
                train_final.loc[s.index, "total_sales"],
                method="pearson"
            )
        )
    )
    .sort_values("mean_sales", ascending=False)
    .reset_index()
)

print(format_price_profile.to_string(index=False))

In [ ]:
# Aim: Quantify how the sales-price slope differs across store formats.

format_price_slopes = []

for store_format, group in train_final.groupby("store_format"):

    x_price = group["product_price"].to_numpy()
    y_sales = group["total_sales"].to_numpy()

    slope, intercept = np.polyfit(x_price, y_sales, deg=1)

    format_price_slopes.append({
        "store_format": store_format,
        "observations": len(group),
        "price_slope": slope,
        "intercept": intercept,
        "price_range": group["product_price"].max() - group["product_price"].min()
    })

format_price_slopes = (
    pd.DataFrame(format_price_slopes)
    .sort_values("price_slope", ascending=False)
    .reset_index(drop=True)
)

print(format_price_slopes.to_string(index=False))

In [ ]:
# Aim: Quantify how the sales-price slope varies across product categories.

category_price_slopes = []

for category, group in train_final.groupby("product_category_clean"):

    x_price = group["product_price"].to_numpy()
    y_sales = group["total_sales"].to_numpy()

    slope, intercept = np.polyfit(x_price, y_sales, deg=1)

    category_price_slopes.append({
        "product_category_clean": category,
        "observations": len(group),
        "price_slope": slope,
        "intercept": intercept,
        "price_range": group["product_price"].max() - group["product_price"].min()
    })

category_price_slopes = (
    pd.DataFrame(category_price_slopes)
    .sort_values("price_slope", ascending=False)
    .reset_index(drop=True)
)

print(category_price_slopes.to_string(index=False))

In [ ]:
# Aim: Measure how the sales-price relationship varies across compact product families.

family_price_data = X_poly.copy()
family_price_data["total_sales"] = train_final["total_sales"]

family_price_profile = (
    family_price_data
    .groupby("product_family")
    .apply(
        lambda g: pd.Series({
            "observations": len(g),
            "mean_price": g["product_price"].mean(),
            "mean_sales": g["total_sales"].mean(),
            "median_sales": g["total_sales"].median(),
            "price_sales_corr": g["product_price"].corr(
                g["total_sales"],
                method="pearson"
            )
        }),
        include_groups=False
    )
    .sort_values("mean_sales", ascending=False)
    .reset_index()
)

print(family_price_profile.to_string(index=False))

In [ ]:
# Aim: Quantify how the linear sales-price slope varies across compact product families.

family_price_slopes = (
    family_price_data
    .groupby("product_family")
    .apply(
        lambda g: pd.Series({
            "observations": len(g),
            "price_slope": np.polyfit(
                g["product_price"],
                g["total_sales"],
                deg=1
            )[0]
        }),
        include_groups=False
    )
    .sort_values("price_slope", ascending=False)
    .reset_index()
)

print(family_price_slopes.to_string(index=False))

print("\nSlope summary:")
print(family_price_slopes["price_slope"].describe())

In [ ]:
# Aim: Check whether compact product families are distributed consistently across store formats.

family_store_composition = (
    family_price_data
    .groupby(["product_family", "store_format"])
    .size()
    .unstack(fill_value=0)
)

family_store_share = family_store_composition.div(
    family_store_composition.sum(axis=1),
    axis=0
)

print("Family x store-format counts:")
print(family_store_composition.to_string())

print("\nFamily x store-format row shares:")
print(family_store_share.round(3).to_string())

Family x store-format counts - quarter 1:
store_format    Corner Shop  Flagship Hypermarket  Standard Supermarket  Superstore
product_family                                                                     
0                        21                    26                   140          17
1                        32                    31                   177          25
2                        33                    25                   143          24
3                        21                    23                   120          17
4                        29                    19                   130          19
5                        21                    28                   143          20
6                        26                    22                   121          26
7                        22                    17                   121          17
8                        23                    20                   104          27

Family x store-format row shares 

In [22]:
# Aim: Evaluate a compact degree-2 polynomial feature space with fold-safe preprocessing and Lasso regularization.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso

poly_categorical = [
    "product_family",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

poly_numeric = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

fold_rmse_poly_lasso = []
oof_poly_lasso = np.zeros(len(X_poly))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_poly), start=1):

    X_tr = X_poly.iloc[train_idx].copy()
    X_val = X_poly.iloc[val_idx].copy()

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ])

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("categorical", categorical_pipeline, poly_categorical),
        ("numeric", numeric_pipeline, poly_numeric)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("polynomial", PolynomialFeatures(
            degree=2,
            include_bias=False
        )),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", Lasso(
            alpha=0.2,
            max_iter=50000,
            random_state=RANDOM_STATE
        ))
    ])

    model.fit(X_tr, y_tr)

    val_pred = model.predict(X_val)

    oof_poly_lasso[val_idx] = val_pred

    fold_rmse = np.sqrt(
        mean_squared_error(y_val, val_pred)
    )

    fold_rmse_poly_lasso.append(fold_rmse)

    print(f"Fold {fold}: {fold_rmse:.6f}")

poly_lasso_mean_rmse = np.mean(fold_rmse_poly_lasso)
poly_lasso_oof_rmse = np.sqrt(
    mean_squared_error(y, oof_poly_lasso)
)

print(f"\nMean fold RMSE: {poly_lasso_mean_rmse:.6f}")
print(f"OOF RMSE: {poly_lasso_oof_rmse:.6f}")

Fold 1: 1195.557624
Fold 2: 1159.461050
Fold 3: 1204.026533
Fold 4: 1237.467867
Fold 5: 1207.534993

Mean fold RMSE: 1200.809613
OOF RMSE:       1201.064430


In [23]:
# Aim: Test targeted product-price by store-format interactions while keeping the remaining feature space at main effects.

from scipy import sparse
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso

fold_rmse_targeted_interaction = []
oof_targeted_interaction = np.zeros(len(X_poly))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_poly), start=1):

    X_tr = X_poly.iloc[train_idx].copy()
    X_val = X_poly.iloc[val_idx].copy()

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    cat_imputer = SimpleImputer(strategy="most_frequent")

    X_tr_cat = cat_imputer.fit_transform(X_tr[poly_categorical])
    X_val_cat = cat_imputer.transform(X_val[poly_categorical])

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )

    X_tr_cat = encoder.fit_transform(X_tr_cat)
    X_val_cat = encoder.transform(X_val_cat)

    num_imputer = SimpleImputer(strategy="median")

    X_tr_num = num_imputer.fit_transform(X_tr[poly_numeric])
    X_val_num = num_imputer.transform(X_val[poly_numeric])

    scaler = StandardScaler()

    X_tr_num = scaler.fit_transform(X_tr_num)
    X_val_num = scaler.transform(X_val_num)

    X_tr_num = sparse.csr_matrix(X_tr_num)
    X_val_num = sparse.csr_matrix(X_val_num)

    X_tr_base = sparse.hstack(
        [X_tr_cat, X_tr_num],
        format="csr"
    )

    X_val_base = sparse.hstack(
        [X_val_cat, X_val_num],
        format="csr"
    )

    format_idx = poly_categorical.index("store_format")
    format_categories = encoder.categories_[format_idx]

    price_idx = poly_numeric.index("product_price")

    cat_start = 0
    for col in poly_categorical[:format_idx]:
        cat_start += len(encoder.categories_[poly_categorical.index(col)])

    format_start = cat_start
    format_end = format_start + len(format_categories)

    tr_format = X_tr_cat[:, format_start:format_end]
    val_format = X_val_cat[:, format_start:format_end]

    tr_price = X_tr_num[:, price_idx].reshape(-1, 1)
    val_price = X_val_num[:, price_idx].reshape(-1, 1)

    tr_interactions = tr_format.multiply(tr_price)
    val_interactions = val_format.multiply(val_price)

    X_tr_final = sparse.hstack(
        [X_tr_base, tr_interactions],
        format="csr"
    )

    X_val_final = sparse.hstack(
        [X_val_base, val_interactions],
        format="csr"
    )

    model = Lasso(
        alpha=1.0,
        max_iter=50000,
        random_state=RANDOM_STATE
    )

    model.fit(X_tr_final, y_tr)

    val_pred = model.predict(X_val_final)

    oof_targeted_interaction[val_idx] = val_pred

    fold_rmse = np.sqrt(
        mean_squared_error(y_val, val_pred)
    )

    fold_rmse_targeted_interaction.append(fold_rmse)

    print(f"Fold {fold}: {fold_rmse:.6f}")

targeted_interaction_mean_rmse = np.mean(
    fold_rmse_targeted_interaction
)

targeted_interaction_oof_rmse = np.sqrt(
    mean_squared_error(y, oof_targeted_interaction)
)

print(f"\nMean fold RMSE: {targeted_interaction_mean_rmse:.6f}")
print(f"OOF RMSE:       {targeted_interaction_oof_rmse:.6f}")

Fold 1: 1062.832135
Fold 2: 1054.226197
Fold 3: 1087.883998
Fold 4: 1113.506064
Fold 5: 1061.306737

Mean fold RMSE: 1075.951026
OOF RMSE:       1076.171708


In [24]:
# Aim: Test a focused range of Lasso regularization strengths for the targeted price-by-store-format interaction.

alpha_values = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0]

alpha_results = []

for alpha in alpha_values:

    fold_rmses = []
    oof_pred_alpha = np.zeros(len(X_poly))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_poly), start=1):

        X_tr = X_poly.iloc[train_idx].copy()
        X_val = X_poly.iloc[val_idx].copy()

        y_tr = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        cat_imputer = SimpleImputer(strategy="most_frequent")

        X_tr_cat = cat_imputer.fit_transform(X_tr[poly_categorical])
        X_val_cat = cat_imputer.transform(X_val[poly_categorical])

        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )

        X_tr_cat = encoder.fit_transform(X_tr_cat)
        X_val_cat = encoder.transform(X_val_cat)

        num_imputer = SimpleImputer(strategy="median")

        X_tr_num = num_imputer.fit_transform(X_tr[poly_numeric])
        X_val_num = num_imputer.transform(X_val[poly_numeric])

        scaler = StandardScaler()

        X_tr_num = scaler.fit_transform(X_tr_num)
        X_val_num = scaler.transform(X_val_num)

        X_tr_num = sparse.csr_matrix(X_tr_num)
        X_val_num = sparse.csr_matrix(X_val_num)

        X_tr_base = sparse.hstack(
            [X_tr_cat, X_tr_num],
            format="csr"
        )

        X_val_base = sparse.hstack(
            [X_val_cat, X_val_num],
            format="csr"
        )

        format_idx = poly_categorical.index("store_format")

        format_start = sum(
            len(encoder.categories_[i])
            for i in range(format_idx)
        )

        format_end = (
            format_start
            + len(encoder.categories_[format_idx])
        )

        tr_format = X_tr_cat[:, format_start:format_end]
        val_format = X_val_cat[:, format_start:format_end]

        price_idx = poly_numeric.index("product_price")

        tr_price = X_tr_num[:, price_idx].reshape(-1, 1)
        val_price = X_val_num[:, price_idx].reshape(-1, 1)

        tr_interactions = tr_format.multiply(tr_price)
        val_interactions = val_format.multiply(val_price)

        X_tr_final = sparse.hstack(
            [X_tr_base, tr_interactions],
            format="csr"
        )

        X_val_final = sparse.hstack(
            [X_val_base, val_interactions],
            format="csr"
        )

        model = Lasso(
            alpha=alpha,
            max_iter=50000,
            random_state=RANDOM_STATE
        )

        model.fit(X_tr_final, y_tr)

        val_pred = model.predict(X_val_final)

        oof_pred_alpha[val_idx] = val_pred

        fold_rmses.append(
            np.sqrt(mean_squared_error(y_val, val_pred))
        )

    mean_rmse = np.mean(fold_rmses)
    oof_rmse = np.sqrt(
        mean_squared_error(y, oof_pred_alpha)
    )

    alpha_results.append({
        "alpha": alpha,
        "mean_fold_rmse": mean_rmse,
        "oof_rmse": oof_rmse
    })

    print(
        f"alpha={alpha:<3} | "
        f"mean fold RMSE={mean_rmse:.6f} | "
        f"OOF RMSE={oof_rmse:.6f}"
    )

alpha_results_df = pd.DataFrame(alpha_results)

print("\nSorted by OOF RMSE:")
display(
    alpha_results_df.sort_values("oof_rmse")
)

alpha=0.1 | mean fold RMSE=1078.671770 | OOF RMSE=1078.886416
alpha=0.2 | mean fold RMSE=1078.312243 | OOF RMSE=1078.527116
alpha=0.5 | mean fold RMSE=1077.292012 | OOF RMSE=1077.508970
alpha=1.0 | mean fold RMSE=1075.951026 | OOF RMSE=1076.171708
alpha=2.0 | mean fold RMSE=1074.301238 | OOF RMSE=1074.524782
alpha=5.0 | mean fold RMSE=1073.461243 | OOF RMSE=1073.671397

Sorted by OOF RMSE:


,alpha,mean_fold_rmse,oof_rmse
5,5.0,1073.461243,1073.671397
4,2.0,1074.301238,1074.524782
3,1.0,1075.951026,1076.171708
2,0.5,1077.292012,1077.508970
1,0.2,1078.312243,1078.527116
0,0.1,1078.671770,1078.886416


In [25]:
# Aim: Extend the targeted Lasso regularization check to determine whether the improving alpha trend continues beyond 5.0.

alpha_values_extended = [10.0, 20.0, 50.0]

extended_alpha_results = []

for alpha in alpha_values_extended:

    fold_rmses = []
    oof_pred_alpha = np.zeros(len(X_poly))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_poly), start=1):

        X_tr = X_poly.iloc[train_idx].copy()
        X_val = X_poly.iloc[val_idx].copy()

        y_tr = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        cat_imputer = SimpleImputer(strategy="most_frequent")

        X_tr_cat = cat_imputer.fit_transform(X_tr[poly_categorical])
        X_val_cat = cat_imputer.transform(X_val[poly_categorical])

        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )

        X_tr_cat = encoder.fit_transform(X_tr_cat)
        X_val_cat = encoder.transform(X_val_cat)

        num_imputer = SimpleImputer(strategy="median")

        X_tr_num = num_imputer.fit_transform(X_tr[poly_numeric])
        X_val_num = num_imputer.transform(X_val[poly_numeric])

        scaler = StandardScaler()

        X_tr_num = scaler.fit_transform(X_tr_num)
        X_val_num = scaler.transform(X_val_num)

        X_tr_num = sparse.csr_matrix(X_tr_num)
        X_val_num = sparse.csr_matrix(X_val_num)

        X_tr_base = sparse.hstack(
            [X_tr_cat, X_tr_num],
            format="csr"
        )

        X_val_base = sparse.hstack(
            [X_val_cat, X_val_num],
            format="csr"
        )

        format_idx = poly_categorical.index("store_format")

        format_start = sum(
            len(encoder.categories_[i])
            for i in range(format_idx)
        )

        format_end = (
            format_start
            + len(encoder.categories_[format_idx])
        )

        tr_format = X_tr_cat[:, format_start:format_end]
        val_format = X_val_cat[:, format_start:format_end]

        price_idx = poly_numeric.index("product_price")

        tr_price = X_tr_num[:, price_idx].reshape(-1, 1)
        val_price = X_val_num[:, price_idx].reshape(-1, 1)

        tr_interactions = tr_format.multiply(tr_price)
        val_interactions = val_format.multiply(val_price)

        X_tr_final = sparse.hstack(
            [X_tr_base, tr_interactions],
            format="csr"
        )

        X_val_final = sparse.hstack(
            [X_val_base, val_interactions],
            format="csr"
        )

        model = Lasso(
            alpha=alpha,
            max_iter=50000,
            random_state=RANDOM_STATE
        )

        model.fit(X_tr_final, y_tr)

        val_pred = model.predict(X_val_final)

        oof_pred_alpha[val_idx] = val_pred

        fold_rmses.append(
            np.sqrt(mean_squared_error(y_val, val_pred))
        )

    mean_rmse = np.mean(fold_rmses)
    oof_rmse = np.sqrt(
        mean_squared_error(y, oof_pred_alpha)
    )

    extended_alpha_results.append({
        "alpha": alpha,
        "mean_fold_rmse": mean_rmse,
        "oof_rmse": oof_rmse
    })

    print(
        f"alpha={alpha:<4} | "
        f"mean fold RMSE={mean_rmse:.6f} | "
        f"OOF RMSE={oof_rmse:.6f}"
    )

extended_alpha_results_df = pd.DataFrame(
    extended_alpha_results
)

print("\nExtended alpha results:")
display(
    extended_alpha_results_df.sort_values("oof_rmse")
)

alpha=10.0 | mean fold RMSE=1075.340756 | OOF RMSE=1075.540991
alpha=20.0 | mean fold RMSE=1082.436738 | OOF RMSE=1082.641079
alpha=50.0 | mean fold RMSE=1119.920240 | OOF RMSE=1120.191323

Extended alpha results:


,alpha,mean_fold_rmse,oof_rmse
0,10.0,1075.340756,1075.540991
1,20.0,1082.436738,1082.641079
2,50.0,1119.920240,1120.191323


In [26]:
# Aim: Identify the available OOF prediction arrays before comparing the targeted Lasso residuals with the CatBoost benchmark.

candidate_oof_names = [
    name for name in globals()
    if name.startswith("oof")
    and isinstance(globals()[name], np.ndarray)
    and globals()[name].shape == (len(y),)
]

print("Available OOF arrays:")
for name in candidate_oof_names:
    print(f"- {name}")

Available OOF arrays:
- oof_poly_lasso
- oof_targeted_interaction
- oof_pred_alpha


In [27]:
# Aim: Reconstruct the frozen CatBoost OOF predictions under the exact locked validation protocol.

from catboost import CatBoostRegressor

catboost_oof = np.zeros(len(X))
catboost_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=lasso_categorical
    )

    val_pred = model.predict(X_val)

    catboost_oof[val_idx] = val_pred

    fold_rmse = np.sqrt(
        mean_squared_error(y_val, val_pred)
    )

    catboost_fold_rmse.append(fold_rmse)

    print(f"Fold {fold}: {fold_rmse:.6f}")

catboost_mean_rmse = np.mean(catboost_fold_rmse)

catboost_oof_rmse = np.sqrt(
    mean_squared_error(y, catboost_oof)
)

print(f"\nMean fold RMSE: {catboost_mean_rmse:.6f}")
print(f"OOF RMSE:       {catboost_oof_rmse:.6f}")
print(f"OOF complete:   {np.all(catboost_oof != 0)}")

Fold 1: 1065.990810
Fold 2: 1053.887357
Fold 3: 1085.198997
Fold 4: 1107.592529
Fold 5: 1061.909675

Mean fold RMSE: 1074.915874
OOF RMSE:       1075.086399
OOF complete:   True


In [28]:
# Aim: Reconstruct and permanently store the best targeted Lasso OOF predictions at alpha=5.0.

best_alpha = 5.0
lasso_best_oof = np.zeros(len(X_poly))
lasso_best_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_poly), start=1):

    X_tr = X_poly.iloc[train_idx].copy()
    X_val = X_poly.iloc[val_idx].copy()

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    cat_imputer = SimpleImputer(strategy="most_frequent")

    X_tr_cat = cat_imputer.fit_transform(X_tr[poly_categorical])
    X_val_cat = cat_imputer.transform(X_val[poly_categorical])

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )

    X_tr_cat = encoder.fit_transform(X_tr_cat)
    X_val_cat = encoder.transform(X_val_cat)

    num_imputer = SimpleImputer(strategy="median")

    X_tr_num = num_imputer.fit_transform(X_tr[poly_numeric])
    X_val_num = num_imputer.transform(X_val[poly_numeric])

    scaler = StandardScaler()

    X_tr_num = scaler.fit_transform(X_tr_num)
    X_val_num = scaler.transform(X_val_num)

    X_tr_num = sparse.csr_matrix(X_tr_num)
    X_val_num = sparse.csr_matrix(X_val_num)

    X_tr_base = sparse.hstack(
        [X_tr_cat, X_tr_num],
        format="csr"
    )

    X_val_base = sparse.hstack(
        [X_val_cat, X_val_num],
        format="csr"
    )

    format_idx = poly_categorical.index("store_format")

    format_start = sum(
        len(encoder.categories_[i])
        for i in range(format_idx)
    )

    format_end = (
        format_start
        + len(encoder.categories_[format_idx])
    )

    tr_format = X_tr_cat[:, format_start:format_end]
    val_format = X_val_cat[:, format_start:format_end]

    price_idx = poly_numeric.index("product_price")

    tr_price = X_tr_num[:, price_idx].reshape(-1, 1)
    val_price = X_val_num[:, price_idx].reshape(-1, 1)

    tr_interactions = tr_format.multiply(tr_price)
    val_interactions = val_format.multiply(val_price)

    X_tr_final = sparse.hstack(
        [X_tr_base, tr_interactions],
        format="csr"
    )

    X_val_final = sparse.hstack(
        [X_val_base, val_interactions],
        format="csr"
    )

    model = Lasso(
        alpha=best_alpha,
        max_iter=50000,
        random_state=RANDOM_STATE
    )

    model.fit(X_tr_final, y_tr)

    val_pred = model.predict(X_val_final)

    lasso_best_oof[val_idx] = val_pred

    fold_rmse = np.sqrt(
        mean_squared_error(y_val, val_pred)
    )

    lasso_best_fold_rmse.append(fold_rmse)

    print(f"Fold {fold}: {fold_rmse:.6f}")

lasso_best_mean_rmse = np.mean(lasso_best_fold_rmse)

lasso_best_oof_rmse = np.sqrt(
    mean_squared_error(y, lasso_best_oof)
)

print(f"\nMean fold RMSE: {lasso_best_mean_rmse:.6f}")
print(f"OOF RMSE:       {lasso_best_oof_rmse:.6f}")
print(f"OOF complete:   {np.all(lasso_best_oof != 0)}")

Fold 1: 1062.113004
Fold 2: 1050.431089
Fold 3: 1084.617718
Fold 4: 1109.970728
Fold 5: 1060.173675

Mean fold RMSE: 1073.461243
OOF RMSE:       1073.671397
OOF complete:   True


In [29]:
# Aim: Compare the prediction and residual structure of the best Lasso and frozen CatBoost models.

catboost_residual = y.to_numpy() - catboost_oof
lasso_residual = y.to_numpy() - lasso_best_oof

prediction_corr = np.corrcoef(
    catboost_oof,
    lasso_best_oof
)[0, 1]

residual_corr = np.corrcoef(
    catboost_residual,
    lasso_residual
)[0, 1]

print(f"Prediction correlation: {prediction_corr:.6f}")
print(f"Residual correlation:   {residual_corr:.6f}")

print("\nOOF RMSE:")
print(f"CatBoost: {np.sqrt(np.mean(catboost_residual ** 2)):.6f}")
print(f"Lasso:    {np.sqrt(np.mean(lasso_residual ** 2)):.6f}")

Prediction correlation: 0.996483
Residual correlation:   0.994831

OOF RMSE:
CatBoost: 1075.086399
Lasso:    1073.671397


In [30]:
# Aim: Measure whether blending CatBoost and the best Lasso produces meaningful OOF improvement.

blend_weights = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

blend_results = []

for lasso_weight in blend_weights:
    catboost_weight = 1.0 - lasso_weight

    blend_pred = (
        catboost_weight * catboost_oof
        + lasso_weight * lasso_best_oof
    )

    rmse = np.sqrt(mean_squared_error(y, blend_pred))

    blend_results.append({
        "lasso_weight": lasso_weight,
        "catboost_weight": catboost_weight,
        "oof_rmse": rmse
    })

blend_results_df = pd.DataFrame(blend_results)

print(blend_results_df.to_string(index=False))

 lasso_weight  catboost_weight    oof_rmse
          0.0              1.0 1075.086399
          0.2              0.8 1073.914182
          0.4              0.6 1073.185945
          0.5              0.5 1072.988623
          0.6              0.4 1072.902592
          0.8              0.2 1073.064477
          1.0              0.0 1073.671397


In [31]:
# Aim: Check whether the Lasso-CatBoost blend improvement is consistent across the locked validation folds.

blend_weights = [0.4, 0.5, 0.6, 0.7]

fold_blend_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    y_val = y.iloc[val_idx].to_numpy()

    for lasso_weight in blend_weights:
        catboost_weight = 1.0 - lasso_weight

        blend_pred = (
            catboost_weight * catboost_oof[val_idx]
            + lasso_weight * lasso_best_oof[val_idx]
        )

        rmse = np.sqrt(mean_squared_error(y_val, blend_pred))

        fold_blend_results.append({
            "fold": fold,
            "lasso_weight": lasso_weight,
            "catboost_weight": catboost_weight,
            "rmse": rmse
        })

fold_blend_results_df = pd.DataFrame(fold_blend_results)

print(
    fold_blend_results_df
    .pivot(index="fold", columns="lasso_weight", values="rmse")
    .round(6)
)

lasso_weight          0.4          0.5          0.6          0.7
fold                                                            
1             1062.867893  1062.413973  1062.091184  1061.899646
2             1051.145386  1050.742611  1050.453238  1050.277360
3             1083.734373  1083.624808  1083.617985  1083.713907
4             1107.351697  1107.540085  1107.827849  1108.214913
5             1059.892182  1059.663192  1059.544549  1059.536290


In [ ]:
# Aim: Identify the existing targeted-Lasso preprocessing objects needed for nested blend validation.

relevant_names = [
    name for name in globals()
    if any(key in name.lower() for key in [
        "lasso",
        "targeted",
        "interaction",
        "encoder",
        "preprocess",
        "poly_numeric",
        "poly_categorical"
    ])
]

for name in sorted(relevant_names):
    obj = globals()[name]
    print(f"{name:<35} {type(obj).__name__}") 

Lasso                               ABCMeta
OneHotEncoder                       type
X_interaction                       DataFrame
encoder                             OneHotEncoder
fold_preprocessor                   ColumnTransformer
fold_rmse_poly_lasso                list
fold_rmse_targeted_interaction      list
interaction_counts                  Series
interaction_midpoint                int
lasso_best_fold_rmse                list
lasso_best_mean_rmse                float64
lasso_best_oof                      ndarray
lasso_best_oof_rmse                 float64
lasso_categorical                   list
lasso_features                      list
lasso_numeric                       list
lasso_oof                           ndarray
lasso_oof_rmse                      float64
lasso_residual                      ndarray
lasso_weight                        float
oof_poly_lasso                      ndarray
oof_targeted_interaction            ndarray
poly_categorical                    list
p

In [33]:
# Aim: Inspect the exact targeted interaction feature representation used by the best Lasso model.

print("X_interaction shape:", X_interaction.shape)
print("\nX_interaction columns:")
print(X_interaction.columns.tolist())

print("\nInteraction midpoint:", interaction_midpoint)

print("\nEncoder:")
print(encoder)

print("\nStored preprocessor:")
print(fold_preprocessor)

X_interaction shape: (6818, 13)

X_interaction columns:
['product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'product_category_clean', 'product_family', 'price_x_store_format']

Interaction midpoint: 3164

Encoder:
OneHotEncoder(handle_unknown='ignore')

Stored preprocessor:
ColumnTransformer(transformers=[('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['product_family', 'fat_content',
                                  'product_category_clean', 'store_code',
                                  'store_size', 'store_location_tier',
                                  '

In [34]:
# Aim: Verify the transformed base feature matrix and the separately constructed price-by-format interaction block.

print("Training interaction matrix shape:", tr_interactions.shape)
print("Validation interaction matrix shape:", val_interactions.shape)

print("\nInteraction midpoint:", interaction_midpoint)

print("\nInteraction block type:", type(tr_interactions).__name__)
print("Interaction block shape:", tr_interactions.shape)

print("\nEncoder output feature count:", len(encoder.get_feature_names_out()))
print("\nFirst 15 encoded feature names:")
print(encoder.get_feature_names_out()[:15])

Training interaction matrix shape: (5455, 4)
Validation interaction matrix shape: (1363, 4)

Interaction midpoint: 3164

Interaction block type: csr_matrix
Interaction block shape: (5455, 4)

Encoder output feature count: 75

First 15 encoded feature names:
['x0_0' 'x0_1' 'x0_2' 'x0_3' 'x0_4' 'x0_5' 'x0_6' 'x0_7' 'x0_8' 'x0_9'
 'x0_A' 'x0_B' 'x0_C' 'x0_D' 'x0_E']


In [35]:
# Aim: Inspect the actual price-by-store-format interaction values to reproduce the targeted Lasso transformation exactly.

interaction_dense = tr_interactions.toarray()

print("Interaction matrix shape:", interaction_dense.shape)

print("\nFirst 5 rows:")
print(interaction_dense[:5])

print("\nColumn means:")
print(interaction_dense.mean(axis=0))

print("\nColumn standard deviations:")
print(interaction_dense.std(axis=0))

print("\nNon-zero counts:")
print((interaction_dense != 0).sum(axis=0))

print("\nColumn maximums:")
print(interaction_dense.max(axis=0))

Interaction matrix shape: (5455, 4)

First 5 rows:
[[ 0.          0.         -0.94011228  0.        ]
 [ 0.          0.         -1.56640486  0.        ]
 [ 0.          0.         -1.57755452  0.        ]
 [ 0.          1.14423783  0.          0.        ]
 [ 0.          0.          0.          0.0367576 ]]

Column means:
[-0.00762918 -0.00200644  0.00895212  0.0006835 ]

Column standard deviations:
[0.3585936  0.31685814 0.81526075 0.32591198]

Non-zero counts:
[ 688  588 3597  582]

Column maximums:
[2.08224316 2.06886357 2.1128251  2.05405044]


In [36]:
# Aim: Perform nested 5-fold confirmation of the CatBoost-Lasso blend using only inner-fold data to select the blend weight.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso
from catboost import CatBoostRegressor

blend_weights = [0.4, 0.5, 0.6, 0.7]

outer_results = []

for outer_fold, (outer_train_idx, outer_val_idx) in enumerate(
    kf.split(X), start=1
):
    X_outer_train = X.iloc[outer_train_idx].copy()
    y_outer_train = y.iloc[outer_train_idx].copy()

    X_outer_val = X.iloc[outer_val_idx].copy()
    y_outer_val = y.iloc[outer_val_idx].copy()

    X_outer_train_poly = X_outer_train.copy()
    X_outer_val_poly = X_outer_val.copy()

    X_outer_train_poly["product_family"] = (
        X_outer_train_poly["product_code"]
        .str.replace("PRD-", "", regex=False)
        .str[0]
    )

    X_outer_val_poly["product_family"] = (
        X_outer_val_poly["product_code"]
        .str.replace("PRD-", "", regex=False)
        .str[0]
    )

    inner_kf = KFold(
        n_splits=4,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    inner_oof_cat = np.zeros(len(X_outer_train))
    inner_oof_lasso = np.zeros(len(X_outer_train))

    for inner_train_idx, inner_val_idx in inner_kf.split(X_outer_train_poly):

        X_inner_train = X_outer_train_poly.iloc[inner_train_idx].copy()
        X_inner_val = X_outer_train_poly.iloc[inner_val_idx].copy()

        y_inner_train = y_outer_train.iloc[inner_train_idx]
        y_inner_val = y_outer_train.iloc[inner_val_idx]

        # CatBoost
        cat_model = CatBoostRegressor(
            boosting_type="Ordered",
            iterations=300,
            learning_rate=0.025,
            depth=4,
            l2_leaf_reg=1,
            random_strength=1.0,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False
        )

        cat_model.fit(
            X_inner_train[lasso_features],
            y_inner_train,
            cat_features=lasso_categorical
        )

        inner_oof_cat[inner_val_idx] = cat_model.predict(
            X_inner_val[lasso_features]
        )

        # Lasso base preprocessing
        categorical = poly_categorical
        numeric = poly_numeric

        inner_preprocessor = ColumnTransformer(
            transformers=[
                (
                    "categorical",
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(strategy="most_frequent")
                        ),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore")
                        )
                    ]),
                    categorical
                ),
                (
                    "numeric",
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(strategy="median")
                        ),
                        (
                            "scaler",
                            StandardScaler()
                        )
                    ]),
                    numeric
                )
            ]
        )

        inner_base_train = inner_preprocessor.fit_transform(
            X_inner_train
        )
        inner_base_val = inner_preprocessor.transform(
            X_inner_val
        )

        # Fold-specific standardized price
        price_imputer = SimpleImputer(strategy="median")

        price_train = price_imputer.fit_transform(
            X_inner_train[["product_price"]]
        )

        price_val = price_imputer.transform(
            X_inner_val[["product_price"]]
        )

        price_scaler = StandardScaler()

        price_train_scaled = price_scaler.fit_transform(price_train)
        price_val_scaled = price_scaler.transform(price_val)

        # Fold-specific store-format indicators
        format_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )

        format_train = format_encoder.fit_transform(
            X_inner_train[["store_format"]]
        )

        format_val = format_encoder.transform(
            X_inner_val[["store_format"]]
        )

        # Price × store-format interaction
        inner_interaction_train = (
            format_train.multiply(price_train_scaled)
        )

        inner_interaction_val = (
            format_val.multiply(price_val_scaled)
        )

        inner_lasso_train = np.hstack([
            inner_base_train.toarray(),
            inner_interaction_train.toarray()
        ])

        inner_lasso_val = np.hstack([
            inner_base_val.toarray(),
            inner_interaction_val.toarray()
        ])

        lasso_model = Lasso(
            alpha=5.0,
            max_iter=50000
        )

        lasso_model.fit(
            inner_lasso_train,
            y_inner_train
        )

        inner_oof_lasso[inner_val_idx] = lasso_model.predict(
            inner_lasso_val
        )

    # Select blend weight using inner OOF predictions only
    inner_weight_rmse = {}

    for lasso_weight in blend_weights:
        cat_weight = 1.0 - lasso_weight

        inner_blend = (
            cat_weight * inner_oof_cat
            + lasso_weight * inner_oof_lasso
        )

        inner_weight_rmse[lasso_weight] = np.sqrt(
            mean_squared_error(
                y_outer_train,
                inner_blend
            )
        )

    selected_weight = min(
        inner_weight_rmse,
        key=inner_weight_rmse.get
    )

    # Retrain CatBoost on complete outer training set
    outer_cat_model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    )

    outer_cat_model.fit(
        X_outer_train[lasso_features],
        y_outer_train,
        cat_features=lasso_categorical
    )

    outer_cat_pred = outer_cat_model.predict(
        X_outer_val[lasso_features]
    )

    # Retrain Lasso on complete outer training set
    outer_preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="most_frequent")
                    ),
                    (
                        "onehot",
                        OneHotEncoder(handle_unknown="ignore")
                    )
                ]),
                poly_categorical
            ),
            (
                "numeric",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]),
                poly_numeric
            )
        ]
    )

    outer_base_train = outer_preprocessor.fit_transform(
        X_outer_train_poly
    )
    outer_base_val = outer_preprocessor.transform(
        X_outer_val_poly
    )

    outer_price_imputer = SimpleImputer(strategy="median")

    outer_price_train = outer_price_imputer.fit_transform(
        X_outer_train_poly[["product_price"]]
    )
    outer_price_val = outer_price_imputer.transform(
        X_outer_val_poly[["product_price"]]
    )

    outer_price_scaler = StandardScaler()

    outer_price_train_scaled = outer_price_scaler.fit_transform(
        outer_price_train
    )
    outer_price_val_scaled = outer_price_scaler.transform(
        outer_price_val
    )

    outer_format_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )

    outer_format_train = outer_format_encoder.fit_transform(
        X_outer_train_poly[["store_format"]]
    )
    outer_format_val = outer_format_encoder.transform(
        X_outer_val_poly[["store_format"]]
    )

    outer_interaction_train = (
        outer_format_train.multiply(outer_price_train_scaled)
    )
    outer_interaction_val = (
        outer_format_val.multiply(outer_price_val_scaled)
    )

    outer_lasso_train = np.hstack([
        outer_base_train.toarray(),
        outer_interaction_train.toarray()
    ])

    outer_lasso_val = np.hstack([
        outer_base_val.toarray(),
        outer_interaction_val.toarray()
    ])

    outer_lasso_model = Lasso(
        alpha=5.0,
        max_iter=50000
    )

    outer_lasso_model.fit(
        outer_lasso_train,
        y_outer_train
    )

    outer_lasso_pred = outer_lasso_model.predict(
        outer_lasso_val
    )

    # Final untouched outer-fold evaluation
    outer_blend_pred = (
        (1.0 - selected_weight) * outer_cat_pred
        + selected_weight * outer_lasso_pred
    )

    outer_rmse = np.sqrt(
        mean_squared_error(
            y_outer_val,
            outer_blend_pred
        )
    )

    outer_results.append({
        "outer_fold": outer_fold,
        "selected_lasso_weight": selected_weight,
        "inner_best_rmse": inner_weight_rmse[selected_weight],
        "outer_blend_rmse": outer_rmse,
        "catboost_rmse": np.sqrt(
            mean_squared_error(y_outer_val, outer_cat_pred)
        ),
        "lasso_rmse": np.sqrt(
            mean_squared_error(y_outer_val, outer_lasso_pred)
        )
    })

    print(
        f"Outer fold {outer_fold}: "
        f"weight={selected_weight:.1f}, "
        f"blend={outer_rmse:.6f}"
    )

nested_blend_results = pd.DataFrame(outer_results)

print("\nNested validation results:")
print(nested_blend_results.to_string(index=False))

print(
    f"\nNested mean blend RMSE: "
    f"{nested_blend_results['outer_blend_rmse'].mean():.6f}"
)

Outer fold 1: weight=0.7, blend=1061.899646
Outer fold 2: weight=0.6, blend=1050.453238
Outer fold 3: weight=0.5, blend=1083.624808
Outer fold 4: weight=0.5, blend=1107.540085
Outer fold 5: weight=0.5, blend=1059.663192

Nested validation results:
 outer_fold  selected_lasso_weight  inner_best_rmse  outer_blend_rmse  catboost_rmse  lasso_rmse
          1                    0.7      1076.539030       1061.899646    1065.990810 1062.113004
          2                    0.6      1077.530173       1050.453238    1053.887357 1050.431089
          3                    0.5      1069.390939       1083.624808    1085.198997 1084.617718
          4                    0.5      1064.357321       1107.540085    1107.592529 1109.970728
          5                    0.5      1076.072414       1059.663192    1061.909675 1060.173675

Nested mean blend RMSE: 1072.636194


In [39]:
# Aim: Compare valid stored OOF candidates by prediction and residual correlation.

candidate_oof = {
    "CatBoost": catboost_oof,
    "Lasso_best": lasso_best_oof,
    "Lasso_plain": lasso_oof,
    "Targeted_interaction": oof_targeted_interaction,
}

prediction_corr = pd.DataFrame(candidate_oof).corr()

residuals = pd.DataFrame({
    name: y.to_numpy() - pred
    for name, pred in candidate_oof.items()
})

residual_corr = residuals.corr()

print("Prediction correlation:")
print(prediction_corr.round(6))

print("\nResidual correlation:")
print(residual_corr.round(6))

Prediction correlation:
                      CatBoost  Lasso_best  Lasso_plain  Targeted_interaction
CatBoost              1.000000    0.996483     0.961179              0.995091
Lasso_best            0.996483    1.000000     0.964785              0.998676
Lasso_plain           0.961179    0.964785     1.000000              0.961939
Targeted_interaction  0.995091    0.998676     0.961939              1.000000

Residual correlation:
                      CatBoost  Lasso_best  Lasso_plain  Targeted_interaction
CatBoost              1.000000    0.994831     0.948311              0.992672
Lasso_best            0.994831    1.000000     0.953735              0.997873
Lasso_plain           0.948311    0.953735     1.000000              0.948455
Targeted_interaction  0.992672    0.997873     0.948455              1.000000


In [40]:
# Aim: Test the stability of the Lasso-CatBoost blend across the validated weight region.

blend_weights = np.arange(0.45, 0.71, 0.05)

blend_results = []

for lasso_weight in blend_weights:
    catboost_weight = 1.0 - lasso_weight

    blended_pred = (
        lasso_weight * lasso_best_oof
        + catboost_weight * catboost_oof
    )

    rmse = np.sqrt(mean_squared_error(y, blended_pred))

    blend_results.append({
        "lasso_weight": lasso_weight,
        "catboost_weight": catboost_weight,
        "oof_rmse": rmse
    })

blend_results = pd.DataFrame(blend_results)

print(blend_results.to_string(index=False))

 lasso_weight  catboost_weight    oof_rmse
         0.45             0.55 1073.073376
         0.50             0.50 1072.988623
         0.55             0.45 1072.931694
         0.60             0.40 1072.902592
         0.65             0.35 1072.901321
         0.70             0.30 1072.927880


In [41]:
# Aim: Test whether product price has store-specific effects beyond the broader store-format interaction.

from scipy import sparse

price_store_oof = np.zeros(len(X))
price_store_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_tr = X_poly.iloc[train_idx].copy()
    X_val = X_poly.iloc[val_idx].copy()
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ])

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("cat", categorical_transformer, poly_categorical),
        ("num", numeric_transformer, poly_numeric)
    ])

    X_tr_encoded = preprocessor.fit_transform(X_tr)
    X_val_encoded = preprocessor.transform(X_val)

    encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]

    cat_feature_names = encoder.get_feature_names_out(poly_categorical)

    numeric_feature_names = np.array(poly_numeric)

    feature_names = np.concatenate([
        cat_feature_names,
        numeric_feature_names
    ])

    price_idx = np.where(feature_names == "product_price")[0][0]

    store_indices = np.where(
        np.char.startswith(feature_names.astype(str), "store_code_")
    )[0]

    X_tr_interaction = sparse.hstack([
        X_tr_encoded,
        X_tr_encoded[:, price_idx].multiply(
            X_tr_encoded[:, store_indices]
        )
    ], format="csr")

    X_val_interaction = sparse.hstack([
        X_val_encoded,
        X_val_encoded[:, price_idx].multiply(
            X_val_encoded[:, store_indices]
        )
    ], format="csr")

    model = Lasso(
        alpha=5.0,
        max_iter=50000,
        random_state=RANDOM_STATE
    )

    model.fit(X_tr_interaction, y_tr)

    pred = model.predict(X_val_interaction)

    price_store_oof[val_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y_val, pred))
    price_store_fold_rmse.append(fold_rmse)

    print(f"Fold {fold}: {fold_rmse:.6f}")

print(f"\nMean fold RMSE: {np.mean(price_store_fold_rmse):.6f}")
print(f"OOF RMSE: {np.sqrt(mean_squared_error(y, price_store_oof)):.6f}")

Fold 1: 1065.599346
Fold 2: 1050.636647
Fold 3: 1086.036286
Fold 4: 1109.319974
Fold 5: 1061.402001

Mean fold RMSE: 1074.598851
OOF RMSE: 1074.797177


In [42]:
# Aim: Test whether product price has category-specific effects beyond the additive category effect.

price_category_oof = np.zeros(len(X))
price_category_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_tr = X_poly.iloc[train_idx].copy()
    X_val = X_poly.iloc[val_idx].copy()
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ])

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("cat", categorical_transformer, poly_categorical),
        ("num", numeric_transformer, poly_numeric)
    ])

    X_tr_encoded = preprocessor.fit_transform(X_tr)
    X_val_encoded = preprocessor.transform(X_val)

    encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]

    cat_feature_names = encoder.get_feature_names_out(poly_categorical)
    numeric_feature_names = np.array(poly_numeric)

    feature_names = np.concatenate([
        cat_feature_names,
        numeric_feature_names
    ])

    price_idx = np.where(feature_names == "product_price")[0][0]

    category_indices = np.where(
        np.char.startswith(
            feature_names.astype(str),
            "product_category_clean_"
        )
    )[0]

    X_tr_interaction = sparse.hstack([
        X_tr_encoded,
        X_tr_encoded[:, price_idx].multiply(
            X_tr_encoded[:, category_indices]
        )
    ], format="csr")

    X_val_interaction = sparse.hstack([
        X_val_encoded,
        X_val_encoded[:, price_idx].multiply(
            X_val_encoded[:, category_indices]
        )
    ], format="csr")

    model = Lasso(
        alpha=5.0,
        max_iter=50000,
        random_state=RANDOM_STATE
    )

    model.fit(X_tr_interaction, y_tr)

    pred = model.predict(X_val_interaction)

    price_category_oof[val_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y_val, pred))
    price_category_fold_rmse.append(fold_rmse)

    print(f"Fold {fold}: {fold_rmse:.6f}")

print(f"\nMean fold RMSE: {np.mean(price_category_fold_rmse):.6f}")
print(f"OOF RMSE: {np.sqrt(mean_squared_error(y, price_category_oof)):.6f}")

Fold 1: 1120.823259
Fold 2: 1102.008204
Fold 3: 1137.585735
Fold 4: 1168.403344
Fold 5: 1114.471990

Mean fold RMSE: 1128.658506
OOF RMSE: 1128.887905


In [43]:
# Aim: Test whether product price has family-specific effects across the structured product-code representation.

price_family_oof = np.zeros(len(X))
price_family_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_tr = X_poly.iloc[train_idx].copy()
    X_val = X_poly.iloc[val_idx].copy()
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ])

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("cat", categorical_transformer, poly_categorical),
        ("num", numeric_transformer, poly_numeric)
    ])

    X_tr_encoded = preprocessor.fit_transform(X_tr)
    X_val_encoded = preprocessor.transform(X_val)

    encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]

    cat_feature_names = encoder.get_feature_names_out(poly_categorical)
    numeric_feature_names = np.array(poly_numeric)

    feature_names = np.concatenate([
        cat_feature_names,
        numeric_feature_names
    ])

    price_idx = np.where(feature_names == "product_price")[0][0]

    family_indices = np.where(
        np.char.startswith(
            feature_names.astype(str),
            "product_family_"
        )
    )[0]

    X_tr_interaction = sparse.hstack([
        X_tr_encoded,
        X_tr_encoded[:, price_idx].multiply(
            X_tr_encoded[:, family_indices]
        )
    ], format="csr")

    X_val_interaction = sparse.hstack([
        X_val_encoded,
        X_val_encoded[:, price_idx].multiply(
            X_val_encoded[:, family_indices]
        )
    ], format="csr")

    model = Lasso(
        alpha=5.0,
        max_iter=50000,
        random_state=RANDOM_STATE
    )

    model.fit(X_tr_interaction, y_tr)

    pred = model.predict(X_val_interaction)

    price_family_oof[val_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y_val, pred))
    price_family_fold_rmse.append(fold_rmse)

    print(f"Fold {fold}: {fold_rmse:.6f}")

print(f"\nMean fold RMSE: {np.mean(price_family_fold_rmse):.6f}")
print(f"OOF RMSE: {np.sqrt(mean_squared_error(y, price_family_oof)):.6f}")

Fold 1: 1121.981852
Fold 2: 1102.354552
Fold 3: 1137.723330
Fold 4: 1167.749617
Fold 5: 1113.536892

Mean fold RMSE: 1128.669249
OOF RMSE: 1128.893849


In [45]:
# Aim: Train the final targeted Lasso model using identical train and test feature representations.

X_full = X_poly.copy()

X_test_poly = X_test.copy()
X_test_poly["product_family"] = (
    X_test_poly["product_code"]
    .str.replace("PRD-", "", regex=False)
    .str[0]
)

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

final_lasso_preprocessor = ColumnTransformer([
    ("cat", categorical_transformer, poly_categorical),
    ("num", numeric_transformer, poly_numeric)
])

X_full_encoded = final_lasso_preprocessor.fit_transform(X_full)
X_test_encoded = final_lasso_preprocessor.transform(X_test_poly)

encoder = (
    final_lasso_preprocessor
    .named_transformers_["cat"]
    .named_steps["onehot"]
)

cat_feature_names = encoder.get_feature_names_out(poly_categorical)
numeric_feature_names = np.array(poly_numeric)

feature_names = np.concatenate([
    cat_feature_names,
    numeric_feature_names
])

price_idx = np.where(feature_names == "product_price")[0][0]

format_indices = np.where(
    np.char.startswith(
        feature_names.astype(str),
        "store_format_"
    )
)[0]

X_full_final = sparse.hstack([
    X_full_encoded,
    X_full_encoded[:, price_idx].multiply(
        X_full_encoded[:, format_indices]
    )
], format="csr")

X_test_final = sparse.hstack([
    X_test_encoded,
    X_test_encoded[:, price_idx].multiply(
        X_test_encoded[:, format_indices]
    )
], format="csr")

final_lasso = Lasso(
    alpha=5.0,
    max_iter=50000,
    random_state=RANDOM_STATE
)

final_lasso.fit(X_full_final, y)

lasso_test_pred = final_lasso.predict(X_test_final)

print("Training shape:", X_full_final.shape)
print("Test shape:", X_test_final.shape)
print("Prediction mean:", lasso_test_pred.mean())
print("Prediction median:", np.median(lasso_test_pred))
print("Prediction min:", lasso_test_pred.min())
print("Prediction max:", lasso_test_pred.max())

Training shape: (6818, 83)
Test shape: (1705, 83)
Prediction mean: 2197.4318196743848
Prediction median: 2084.3691274054777
Prediction min: 42.23134751795942
Prediction max: 6611.070354038231


In [46]:
# Aim: Train the frozen CatBoost champion on all available training data and predict the test set.

final_catboost = CatBoostRegressor(
    boosting_type="Ordered",
    iterations=300,
    learning_rate=0.025,
    depth=4,
    l2_leaf_reg=1,
    random_strength=1.0,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False
)

final_catboost.fit(
    X,
    y,
    cat_features=lasso_categorical
)

catboost_test_pred = final_catboost.predict(X_test)

print("Prediction mean:", catboost_test_pred.mean())
print("Prediction median:", np.median(catboost_test_pred))
print("Prediction min:", catboost_test_pred.min())
print("Prediction max:", catboost_test_pred.max())

Prediction mean: 2210.526817627066
Prediction median: 2139.8934926324473
Prediction min: 7.296992603987746
Prediction max: 5938.934543869664


In [47]:
# Aim: Construct the validated Lasso-CatBoost blend while preserving the pure Lasso predictions.

lasso_weight = 0.60
catboost_weight = 0.40

blend_test_pred = (
    lasso_weight * lasso_test_pred
    + catboost_weight * catboost_test_pred
)

print("Lasso prediction mean:", lasso_test_pred.mean())
print("CatBoost prediction mean:", catboost_test_pred.mean())
print("Blend prediction mean:", blend_test_pred.mean())

print("\nLasso prediction range:", lasso_test_pred.min(), "to", lasso_test_pred.max())
print("CatBoost prediction range:", catboost_test_pred.min(), "to", catboost_test_pred.max())
print("Blend prediction range:", blend_test_pred.min(), "to", blend_test_pred.max())

Lasso prediction mean: 2197.4318196743848
CatBoost prediction mean: 2210.526817627066
Blend prediction mean: 2202.6698188554574

Lasso prediction range: 42.23134751795942 to 6611.070354038231
CatBoost prediction range: 7.296992603987746 to 5938.934543869664
Blend prediction range: 28.66971976666318 to 6327.358166066507


In [48]:
# Aim: Compare target and test-prediction quantiles to detect distributional anomalies before submission.

comparison = pd.DataFrame({
    "train_target": y.quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]),
    "lasso": pd.Series(lasso_test_pred).quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]),
    "catboost": pd.Series(catboost_test_pred).quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]),
    "blend": pd.Series(blend_test_pred).quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
})

comparison.index = [
    "1%", "5%", "10%", "25%", "50%",
    "75%", "90%", "95%", "99%"
]

comparison

,train_target,lasso,catboost,blend
1%,81.2017,114.546843,94.419636,107.655769
5%,188.1190,359.250372,334.664440,350.404631
10%,342.0420,528.441847,499.046333,524.757645
25%,834.7925,1226.533434,1246.382289,1229.215177
50%,1790.8900,2084.369127,2139.893493,2098.950565
75%,3092.5875,3032.402714,3117.010801,3061.271729
90%,4557.8410,3942.896250,3930.506958,3951.002951
95%,5557.1910,4289.140218,4050.322205,4193.726956
99%,7261.3893,5865.661852,5871.603298,5868.408719


In [50]:
# Aim: Recover the original test IDs and verify they align exactly with the prediction rows.

test_raw = pd.read_csv("../data/test.csv")

print("Raw test shape:", test_raw.shape)
print("Raw test columns:", test_raw.columns.tolist())
print("Prediction length:", len(lasso_test_pred))

assert len(test_raw) == len(lasso_test_pred)
assert len(test_raw) == len(blend_test_pred)
assert "id" in test_raw.columns

print("\nID alignment check passed.")
print("First 5 IDs:", test_raw["id"].head().tolist())
print("Last 5 IDs:", test_raw["id"].tail().tolist())

Raw test shape: (1705, 12)
Raw test columns: ['id', 'product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format']
Prediction length: 1705

ID alignment check passed.
First 5 IDs: ['row_00009', 'row_00015', 'row_00019', 'row_00020', 'row_00023']
Last 5 IDs: ['row_08483', 'row_08489', 'row_08491', 'row_08504', 'row_08515']


In [51]:
# Aim: Create and validate the final Lasso and Lasso-CatBoost submission files.

submission_lasso = pd.DataFrame({
    "id": test_raw["id"],
    "total_sales": lasso_test_pred
})

submission_blend = pd.DataFrame({
    "id": test_raw["id"],
    "total_sales": blend_test_pred
})

submission_lasso.to_csv(
    "../submissions/submission_lasso_alpha5.csv",
    index=False
)

submission_blend.to_csv(
    "../submissions/submission_lasso60_catboost40.csv",
    index=False
)

for name, submission in [
    ("Lasso α=5", submission_lasso),
    ("Lasso 60% + CatBoost 40%", submission_blend)
]:
    print(f"\n{name}")
    print("Shape:", submission.shape)
    print("Columns:", submission.columns.tolist())
    print("Missing values:")
    print(submission.isna().sum())
    print("Unique IDs:", submission["id"].nunique())
    print("ID order matches raw test:", submission["id"].equals(test_raw["id"]))
    print("Prediction mean:", submission["total_sales"].mean())
    print("Prediction min:", submission["total_sales"].min())
    print("Prediction max:", submission["total_sales"].max())

print("\nSubmission files created successfully.")


Lasso α=5
Shape: (1705, 2)
Columns: ['id', 'total_sales']
Missing values:
id             0
total_sales    0
dtype: int64
Unique IDs: 1705
ID order matches raw test: True
Prediction mean: 2197.4318196743848
Prediction min: 42.23134751795942
Prediction max: 6611.070354038231

Lasso 60% + CatBoost 40%
Shape: (1705, 2)
Columns: ['id', 'total_sales']
Missing values:
id             0
total_sales    0
dtype: int64
Unique IDs: 1705
ID order matches raw test: True
Prediction mean: 2202.6698188554574
Prediction min: 28.66971976666318
Prediction max: 6327.358166066507

Submission files created successfully.


In [52]:
# Aim: Verify the exact filesystem location and existence of the final submission files.

from pathlib import Path

submission_dir = Path("../submissions").resolve()

print("Submission directory:")
print(submission_dir)

print("\nContents:")
for file in sorted(submission_dir.iterdir()):
    print(file.name)

print("\nFinal Lasso exists:", (submission_dir / "submission_lasso_alpha5.csv").exists())
print("Final blend exists:", (submission_dir / "submission_lasso60_catboost40.csv").exists())

Submission directory:
C:\Users\Administrator\Documents\DSN Bootcamp Qualification Hackathon 2026 ML Track\submissions

Contents:
submission_lasso60_catboost40.csv
submission_lasso_alpha5.csv

Final Lasso exists: True
Final blend exists: True
